# DAISY 3 Full-Book Generation in Colab

This notebook builds the full DAISY 3 package from the canonical structured book.json committed in the repository.

The source of truth is build/structured/book.json. It does not require a Google Drive EPUB upload or any EPUB extraction step.

Run the cells in order.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
print('✓ Drive mounted')


In [ ]:
import os

DRIVE_BASE = '/content/drive/MyDrive/daisy-psychology-of-money'
DRIVE_AUDIO_CACHE = os.path.join(DRIVE_BASE, 'cache', 'audio')
DRIVE_OUTPUT = os.path.join(DRIVE_BASE, 'output')
WORKDIR = '/content/Daisy'
REPO_URL = 'https://github.com/tthongbos/Daisy.git'
REPO_REF = 'main'

os.makedirs(DRIVE_AUDIO_CACHE, exist_ok=True)
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

print(f'✓ Drive audio cache: {DRIVE_AUDIO_CACHE}')
print(f'✓ Drive output: {DRIVE_OUTPUT}')
print(f'✓ repo directory: {WORKDIR}')


In [ ]:
import os
import subprocess

if os.path.exists(WORKDIR):
    print(f'Repository already exists at {WORKDIR}; updating it.')
    fetch = subprocess.run(['git', '-C', WORKDIR, 'fetch', 'origin', REPO_REF], capture_output=True, text=True)
    if fetch.returncode != 0:
        raise RuntimeError(fetch.stderr)
    checkout = subprocess.run(['git', '-C', WORKDIR, 'checkout', REPO_REF], capture_output=True, text=True)
    if checkout.returncode != 0:
        raise RuntimeError(checkout.stderr)
    pull = subprocess.run(['git', '-C', WORKDIR, 'pull', '--ff-only', 'origin', REPO_REF], capture_output=True, text=True)
    if pull.returncode != 0:
        raise RuntimeError(pull.stderr)
else:
    result = subprocess.run(['git', 'clone', '--branch', REPO_REF, '--single-branch', REPO_URL, WORKDIR], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr)

os.chdir(WORKDIR)
print(f'✓ repo ready: {WORKDIR}')


In [ ]:
import subprocess

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
print('✓ ffmpeg installed and project dependencies are ready')


In [ ]:
import json
import os

BOOK_JSON = os.path.join(WORKDIR, 'build', 'structured', 'book.json')

if not os.path.isfile(BOOK_JSON):
    raise FileNotFoundError(f'Canonical structured book not found: {BOOK_JSON}')

with open(BOOK_JSON, 'r', encoding='utf-8') as handle:
    book = json.load(handle)

if not isinstance(book, dict):
    raise TypeError('book.json root must be a JSON object')

metadata = book.get('metadata')
sections = book.get('sections')
if not isinstance(metadata, dict):
    raise TypeError('book.json metadata must be an object')
if not isinstance(sections, list) or not sections:
    raise ValueError('book.json sections must be a non-empty list')

section_ids = []
for entry in sections:
    if not isinstance(entry, dict):
        raise TypeError('Each section must be an object')
    section_id = entry.get('id')
    if not section_id or not str(section_id).strip():
        raise ValueError('Every section must have a non-empty id')
    section_ids.append(str(section_id))

print(f'✓ title: {metadata.get("title", "") }')
print(f'✓ author: {metadata.get("author", "") }')
print(f'✓ sections: {len(section_ids)}')
print('')
for section_id in section_ids:
    print(section_id)

if len(section_ids) != 21:
    print(f'⚠ Warning: this repository has {len(section_ids)} sections; expected 21 for the current book.')


In [ ]:
import os

build_root = os.path.join(WORKDIR, 'build')
audio_dir = os.path.join(build_root, 'audio')

os.makedirs(build_root, exist_ok=True)
os.makedirs(DRIVE_AUDIO_CACHE, exist_ok=True)

if os.path.islink(audio_dir):
    real_audio_path = os.path.realpath(audio_dir)
    if real_audio_path != os.path.realpath(DRIVE_AUDIO_CACHE):
        os.unlink(audio_dir)
        os.symlink(DRIVE_AUDIO_CACHE, audio_dir)
    print(f'✓ audio cache is active: {audio_dir} -> {os.path.realpath(audio_dir)}')
elif os.path.isdir(audio_dir):
    if not os.listdir(audio_dir):
        os.rmdir(audio_dir)
        os.symlink(DRIVE_AUDIO_CACHE, audio_dir)
        print(f'✓ audio cache symlink created: {audio_dir} -> {DRIVE_AUDIO_CACHE}')
    else:
        print(f'⚠ existing non-empty local audio directory preserved at {audio_dir}; not deleting it.')
else:
    os.symlink(DRIVE_AUDIO_CACHE, audio_dir)
    print(f'✓ audio cache symlink created: {audio_dir} -> {DRIVE_AUDIO_CACHE}')


In [ ]:
from google.colab import userdata
import os

azure_key = userdata.get('AZURE_SPEECH_KEY')
azure_region = userdata.get('AZURE_SPEECH_REGION')
azure_voice = userdata.get('AZURE_SPEECH_VOICE') or 'vi-VN-HoaiMyNeural'

if azure_key and azure_region:
    os.environ['AZURE_SPEECH_KEY'] = azure_key
    os.environ['AZURE_SPEECH_REGION'] = azure_region
    os.environ['AZURE_SPEECH_VOICE'] = azure_voice
    print('✓ Azure credentials loaded from Colab Secrets')
    print(f'  region: {azure_region}')
    print(f'  voice: {azure_voice}')
else:
    print('⚠ Azure credentials missing. Dry-run still works, but the full build will be skipped.')


In [ ]:
import subprocess

print('Running pytest -q before Azure synthesis...')
result = subprocess.run(['pytest', '-q'], cwd=WORKDIR, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'pytest failed with exit code {result.returncode}')
print('✓ pytest passed')


In [ ]:
import subprocess

print('Running full-book dry-run (no Azure cost)...')
result = subprocess.run(['make', 'full-book-dry-run'], cwd=WORKDIR, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'dry-run failed with exit code {result.returncode}')
print('✓ full-book dry-run complete')


In [ ]:
import os
import subprocess

os.chdir(WORKDIR)
if not os.environ.get('AZURE_SPEECH_KEY') or not os.environ.get('AZURE_SPEECH_REGION'):
    print('⚠ Azure credentials unavailable; full build skipped.')
else:
    print('Running make tts-audit...')
    result = subprocess.run(['make', 'tts-audit'], cwd=WORKDIR, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'make tts-audit failed with exit code {result.returncode}')
    print('✓ tts-audit complete')

    print('Running make package-all...')
    result = subprocess.run(['make', 'package-all'], cwd=WORKDIR, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'make package-all failed with exit code {result.returncode}')
    print('✓ full DAISY build complete')


In [ ]:
import glob
import json
import os
import re
import subprocess

build_dir = os.path.join(WORKDIR, 'build', 'daisy')
if not os.path.exists(build_dir):
    print('⚠ No DAISY build artifacts found; build was skipped because Azure credentials were unavailable.')
else:
    print('Verifying generated DAISY resources...')
    with open(BOOK_JSON, 'r', encoding='utf-8') as handle:
        book = json.load(handle)
    sections = book.get('sections', [])
    section_ids = [str(section['id']) for section in sections if isinstance(section, dict) and section.get('id')]

    required_files = ['book.xml', 'book.ncx', 'book.opf']
    for name in required_files:
        file_path = os.path.join(build_dir, name)
        if not os.path.isfile(file_path):
            raise FileNotFoundError(f'Missing required DAISY artifact: {file_path}')
        print(f'✓ found {name}')

    for section_id in section_ids:
        smil_path = os.path.join(build_dir, f'{section_id}.smil')
        mp3_path = os.path.join(build_dir, f'{section_id}.mp3')
        if not os.path.isfile(smil_path):
            raise FileNotFoundError(f'Missing SMIL for section {section_id}: {smil_path}')
        if not os.path.isfile(mp3_path):
            raise FileNotFoundError(f'Missing MP3 for section {section_id}: {mp3_path}')
        print(f'✓ section resources: {section_id}')

    smil_files = sorted(glob.glob(os.path.join(build_dir, '*.smil')))
    npt_matches = []
    colon_clock_matches = []
    for smil_file in smil_files:
        with open(smil_file, 'r', encoding='utf-8') as handle:
            content = handle.read()
        if 'npt=' in content:
            npt_matches.append(smil_file)
        if re.search(r'\d+:\d{2}:\d{2}\.\d{3}', content):
            colon_clock_matches.append(smil_file)
    if npt_matches:
        raise RuntimeError(f'Found npt= in generated SMIL files: {npt_matches}')
    if not colon_clock_matches:
        raise RuntimeError('No colon-clock timestamps found in generated SMIL files.')
    print(f'✓ no npt= found across {len(smil_files)} SMIL files')
    print(f'✓ colon-clock timestamps detected in {len(colon_clock_matches)} SMIL files')

    mp3_files = sorted(glob.glob(os.path.join(build_dir, '*.mp3')))
    if not mp3_files:
        raise FileNotFoundError(f'No MP3 files found in {build_dir}')

    for mp3_file in mp3_files:
        result = subprocess.run([
            'ffprobe', '-v', 'error', '-select_streams', 'a:0',
            '-show_entries', 'stream=codec_name,sample_rate,channels',
            '-of', 'default=noprint_wrappers=1:nokey=1', mp3_file
        ], capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(f'ffprobe failed for {mp3_file}: {result.stderr}')
        values = [item.strip() for item in result.stdout.splitlines() if item.strip()]
        if len(values) < 3:
            raise RuntimeError(f'Incomplete ffprobe output for {mp3_file}: {values!r}')
        codec_name, sample_rate, channels = values[:3]
        if codec_name != 'mp3':
            raise RuntimeError(f'{os.path.basename(mp3_file)} has codec_name={codec_name!r}, expected mp3')
        if sample_rate != '22050':
            raise RuntimeError(f'{os.path.basename(mp3_file)} has sample_rate={sample_rate!r}, expected 22050')
        if channels != '1':
            raise RuntimeError(f'{os.path.basename(mp3_file)} has channels={channels!r}, expected 1')
        print(f'✓ audio_ok: {os.path.basename(mp3_file)} ({codec_name}, {sample_rate}Hz, {channels}ch)')

    validator = subprocess.run(['python', '-m', 'daisy_book.validate_daisy', '--input', 'build/daisy'], cwd=WORKDIR, capture_output=True, text=True)
    print(validator.stdout)
    if validator.returncode != 0:
        print(validator.stderr)
        raise RuntimeError(f'DAISY validation failed with exit code {validator.returncode}')
    print('✓ DAISY validation passed')


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

local_zip = Path(WORKDIR) / 'output' / 'Tam_ly_hoc_ve_tien_DAISY3.zip'
local_checksum = Path(WORKDIR) / 'output' / 'Tam_ly_hoc_ve_tien_DAISY3_sha256sums.txt'

if not local_zip.exists() or not local_checksum.exists():
    print('⚠ No final package artifacts are present; skipping ZIP verification.')
else:
    print('Running ZIP integrity checks...')
    unzip_result = subprocess.run(['unzip', '-t', str(local_zip)], capture_output=True, text=True)
    print(unzip_result.stdout)
    if unzip_result.returncode != 0:
        print(unzip_result.stderr)
        raise RuntimeError(f'ZIP integrity check failed with exit code {unzip_result.returncode}')

    with zipfile.ZipFile(local_zip, 'r') as zf:
        names = set(zf.namelist())
        required_members = {'book.xml', 'book.ncx', 'book.opf'}
        with open(BOOK_JSON, 'r', encoding='utf-8') as handle:
            book = json.load(handle)
        for section in [
            str(entry['id'])
            for entry in book.get('sections', [])
            if isinstance(entry, dict) and entry.get('id')
        ]:
            required_members.add(f'{section}.smil')
            required_members.add(f'{section}.mp3')
        missing = sorted(member for member in required_members if member not in names)
        if missing:
            raise FileNotFoundError(f'Missing entries in ZIP: {missing}')
        print(f'✓ ZIP contains {len(names)} members including all required DAISY files')

    checksum_text = local_checksum.read_text(encoding='utf-8').strip()
    expected_hash, _ = checksum_text.split(None, 1)
    actual_hash = hashlib.sha256(local_zip.read_bytes()).hexdigest()
    if actual_hash != expected_hash:
        raise RuntimeError(f'SHA256 mismatch for {local_zip.name}: expected {expected_hash}, got {actual_hash}')
    print(f'✓ SHA256 matches checksum file: {actual_hash}')

    os.makedirs(DRIVE_OUTPUT, exist_ok=True)
    drive_zip = os.path.join(DRIVE_OUTPUT, local_zip.name)
    drive_checksum = os.path.join(DRIVE_OUTPUT, local_checksum.name)
    shutil.copy2(local_zip, drive_zip)
    shutil.copy2(local_checksum, drive_checksum)
    drive_hash = hashlib.sha256(Path(drive_zip).read_bytes()).hexdigest()
    if drive_hash != actual_hash:
        raise RuntimeError(f'Drive copy hash mismatch: {drive_hash} != {actual_hash}')
    print(f'✓ copied final ZIP and checksum to {DRIVE_OUTPUT}')

    print('DAISY full-book build verified')
    print('')
    print('Source:')
    print(f'  {BOOK_JSON}')
    with open(BOOK_JSON, 'r', encoding='utf-8') as handle:
        book = json.load(handle)
    section_ids = [
        str(entry['id'])
        for entry in book.get('sections', [])
        if isinstance(entry, dict) and entry.get('id')
    ]
    print('Sections:')
    print(f'  {len(section_ids)}')
    print('Audio cache:')
    print(f'  {DRIVE_AUDIO_CACHE}')
    print('Package:')
    print(f'  {drive_zip}')
    print('SHA256:')
    print(f'  {actual_hash}')
